In [ ]:
import pickle
import networkx as nx
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import pandas as pd
import ast
import seaborn as sns
import sys

from src.functions import *
from src.ml_models import *

In [ ]:
sys.path.append("src")

In [ ]:
# If clipping is enabled, removes all paths not represented by the dataset from being
# evaluated. Otherwise, all out-of-distribution defects are omitted.

CLIPPING = False

# The following code is used to select the model architecture

# model_version = "rnn"      # sRNN
model_version = "gru"  # GRU

In [ ]:
with open("gen/network.pickle", "rb") as f:
    G = pickle.load(f)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
def deduplicate_in_place(items):
    _tmp_list = []
    for item in items:
        if item not in _tmp_list:
            _tmp_list.append(item)
    items[:] = _tmp_list

In [ ]:
df = pd.read_csv("gen/ml_data.csv")
templates = df["defect_path"].unique().tolist()
for i, template in enumerate(templates):
    templates[i] = ast.literal_eval(template)

# Get the set of all possible paths covered by the dataset
in_distribution_paths = []
for template in templates:
    in_distribution_paths.extend(get_path_permutations(template))

if in_distribution_paths[0][0] != "SPAWN":
    in_distribution_paths = [
        ["SPAWN"] + list(path) + ["DETECTION"] for path in in_distribution_paths
    ]

deduplicate_in_place(in_distribution_paths)

# Get the set of all possible paths in the network
paths = list(nx.all_simple_paths(G, source="SPAWN", target="DETECTION"))
paths = [path for path in paths if len(path) > 3]  # Filter out intra-phase defects

print(
    f"Percentage of paths in distribution:"
    f" {100 * len(in_distribution_paths) / len(paths):.2f}%"
)

In [ ]:
mean_defect_cost = df["task_timespent"].mean()
print(f"Mean defect cost: {mean_defect_cost:.2f}")

In [ ]:
for node in G.nodes:
    total_weight = sum([G[node][neighbor]["weight"] for neighbor in G[node]])

    for neighbor in G[node]:
        prob = G[node][neighbor]["weight"] / total_weight
        G[node][neighbor]["logprob"] = np.log(prob)

In [ ]:
def get_path_logprob(path, G=G):
    total_logprob = 0

    if path[0] != "SPAWN":
        path = ["SPAWN"] + list(path) + ["DETECTION"]

    current_node = "SPAWN"
    for node in path[1:]:
        logprob = G[current_node][node]["logprob"]
        total_logprob += logprob
        current_node = node

    return total_logprob

In [ ]:
prob_mass = 0
for path in in_distribution_paths:
    logprob = get_path_logprob(path)
    prob_mass += np.exp(logprob)

print(f"Total probability mass of in-distribution paths: {prob_mass:.2f}")

In [ ]:
model = torch.load(f"gen/model_{model_version}.pt", weights_only=False).to("cpu")
model.eval()

In [ ]:
if CLIPPING:
    paths = list(in_distribution_paths)

In [ ]:
# Implementation of incorrect specification:    5d
IMPLEMENTATION_OFFSET = 5 * 7.5

# Additional round of verification:             5d
VERIFICATION_OFFSET = 5 * 7.5

# Development opportunity cost:                 5d
# SW environment to customer version:           5d
# I&V environment to customer version:          4d
# Emergency meetings:                           3d
# Customer documentation:                       1d
# Customer service and training:                5d
# Avg’d cost of setting up additional release:  5d
CUSTOMER_OFFSET = (5 + 5 + 4 + 3 + 1 + 5 + 5) * 7.5

In [ ]:
def get_path_offset(path):
    offset = 0
    # Apply implementation offset
    if any(node.startswith("Specification") for node in path) and any(
        node.startswith("Implementation") for node in path
    ):
        offset += IMPLEMENTATION_OFFSET

    # Apply verification offset
    if any(node.startswith("Integration") for node in path) and any(
        node.startswith("Verification") for node in path
    ):
        offset += VERIFICATION_OFFSET

    # Apply customer offset
    if any(node.startswith("FieldValidation") for node in path) and any(
        node.startswith("Customer") for node in path
    ):
        offset += CUSTOMER_OFFSET

    return offset


def calculate_mean_cost_baseline(
    G,
    model=model,
    device="cuda",
    paths=paths,
):
    offsets = torch.tensor(
        [get_path_offset(path) for path in paths],
        dtype=torch.float32,
        device=device,
    )

    log_probs = torch.tensor(
        [get_path_logprob(path, G=G) for path in paths],
        dtype=torch.float32,
        device=device,
    )

    path_tensors = [path_to_tensor(path) for path in paths]
    padded_batch = pad_sequence(path_tensors, batch_first=True).to(device)
    lengths = torch.tensor(
        [len(path) for path in paths],
        dtype=torch.float32,
        device="cpu",
    )

    model = model.to(device)
    model.eval()

    with torch.no_grad():
        model_preds = model(padded_batch, lengths).squeeze()

        if model_preds.dim() == 0:
            model_preds = model_preds.unsqueeze(0)

    # Prediction unit is log-adjusted. Exponentiate to get actual prediction values.
    cost_preds = torch.exp(model_preds)

    return cost_preds, offsets, log_probs

In [ ]:
num_parameters = sum(p.numel() for p in model.parameters())
print(f"The {model_version} model has {num_parameters} parameters.")

In [ ]:
cost_preds, base_offsets, log_p_base = calculate_mean_cost_baseline(
    G, device=device, paths=paths
)

print(f"Model version: {model_version}")
print(f"Path set: {'zeta' if CLIPPING else 'Gamma'}")

p_base = torch.exp(log_p_base)
baseline_cost = (p_base * (cost_preds + base_offsets)).sum().item()
print(f"Baseline adjusted mean defect cost: {baseline_cost:.2f}")

for node_i in G.neighbors("SPAWN"):
    G_alt = G.copy()
    if "DETECTION" in G_alt[node_i].keys():
        # Increase detection probability by 25% at node
        G_alt[node_i]["DETECTION"]["weight"] *= 1.25

        # Recalculate transition log probabilities
        for node_j in G_alt.nodes:
            total_weight = sum(
                [G_alt[node_j][neighbor]["weight"] for neighbor in G_alt[node_j]]
            )

            for neighbor in G_alt[node_j]:
                prob = G_alt[node_j][neighbor]["weight"] / total_weight
                G_alt[node_j][neighbor]["logprob"] = np.log(prob)

        log_p_prime = torch.tensor(
            [get_path_logprob(path, G=G_alt) for path in paths],
            dtype=torch.float32,
            device=device,
        )
        p_prime = torch.exp(log_p_prime)

        early_offsets = []
        for path in paths:
            if node_i in path:
                # Truncate path at node_i to simulate early detection
                idx = path.index(node_i)
                early_path = path[: idx + 1] + ["DETECTION"]
                early_offsets.append(get_path_offset(early_path))
            else:
                # If path doesn't cross node_i, it retains its base offset
                early_offsets.append(get_path_offset(path))

        early_offsets = torch.tensor(early_offsets, dtype=torch.float32, device=device)

        escaped_cost = p_prime * (cost_preds + base_offsets)
        caught_cost = (p_base - p_prime) * (cost_preds + early_offsets)

        mean_cost = (escaped_cost + caught_cost).sum().item()
        improvement = baseline_cost - mean_cost

        print(f"{node_i: <22} | {mean_cost: >5.2f} | {improvement: >5.3f}")